In [1]:
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import n_local
from qiskit_algorithms.optimizers import COBYLA, SPSA
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error, ReadoutError
from qiskit_aer.primitives import Estimator
import numpy as np

In [2]:
#Building the frustrated Ising Hamiltonian with transverse field
N = 3; #Number of spins
coeffs = [0.5,0.5]; #[J,h]
H = ( SparsePauliOp.from_list([("ZZI", coeffs[0])]) +
          SparsePauliOp.from_list([("ZIZ", coeffs[0])]) +
          SparsePauliOp.from_list([("IZZ", coeffs[0])]) +
          SparsePauliOp.from_list([("XII", coeffs[1])]) +
          SparsePauliOp.from_list([("IXI", coeffs[1])]) + 
          SparsePauliOp.from_list([("IIX", coeffs[1])]) )

#Ry-CZ Ansatz for the trial wavefunction 
p = N*(3 + 1); #Total number of parameters with reps = 3;
ansatz = n_local(N, rotation_blocks = 'ry', entanglement_blocks = 'cz', reps=3)


In [3]:
#Building the noise model
#(time measured in nanoseconds)
T1 = 100e3; #time for qubit to decay from |1> to |0>
T2 = 80e3; #time for qubit to lose phase coherence

#Gate durations (hpw long each gate takes on hardware)
t_1q = 50    #for single-qubit gate 
t_2q = 300   #for two-qubit gate 
t_meas = 1000  #for measurement

#Gate error rates (probability of depolarizing error per gate)
p_1q = 0.001   # 0.1% error per single-qubit gate
p_2q = 0.01    # 1% error per two-qubit gate

noise_model = NoiseModel()

#Single qubit gate errors
error_1q_depol = depolarizing_error(p_1q, 1) #Depolarizing error
error_1q_thermal = thermal_relaxation_error(T1,T2,t_1q) #Thermal relaxation error
error_1q = error_1q_depol.compose(error_1q_thermal) #Combining all the single qubit errors

#Adding the single qubit errors to the noise model
noise_model.add_all_qubit_quantum_error(error_1q, ['ry', 'rx', 'u']) 

#Two qubit gate errors
error_2q_depol = depolarizing_error(p_2q, 2) #Depolarizing error

#thermal relaxation errors acting on both the qubits in a two qubit gate
error_2q_thermal_q0 = thermal_relaxation_error(T1,T2,t_2q); 
error_2q_thermal_q1 = thermal_relaxation_error(T1,T2,t_2q);
error_2q_thermal = error_2q_thermal_q0.expand(error_2q_thermal_q1);

#Combining all the two qubit errors
error_2q = error_2q_depol.compose(error_2q_thermal);

#Adding the two qubit errors to the noise model
noise_model.add_all_qubit_quantum_error(error_2q,['cx'])

#Readout error
pmeas_01 = 0.02; #p(1|0)
pmeas_10 = 0.04; #p(0|1)
readout_error = ReadoutError([[1 - pmeas_01, pmeas_01],[pmeas_10, 1-pmeas_10]]);

#Adding the r\error to all qubits
for q in range(N):
    noise_model.add_readout_error(readout_error, [q])

In [4]:
#Building the Noisy Estimator
noisy_estimator = Estimator(backend_options={ "noise_model": noise_model, "method": "density_matrix"},run_options={"shots": 1024})

#Noisy Expectation Value Function
def expectation_value(params):
    job = noisy_estimator.run(circuits = [ansatz], observables = [H], parameter_values = [params])
    result = job.result()
    return float(result.values[0].real)

In [5]:
#Running on Noisy VQE using COBYLA
optimizerC = COBYLA(maxiter=500, tol=1e-4)
rngC = np.random.default_rng(42)

best_noisy_energyC = np.inf;
best_noisy_paramsC = None;

for i in range(5): #using 5 random restarts
    x0 = rngC.uniform(-np.pi, np.pi, ansatz.num_parameters)
    result = optimizerC.minimize(fun=expectation_value, x0=x0)
    if result.fun < best_noisy_energyC:
        best_noisy_energyC = result.fun
        best_noisy_parameterC = result.x

# Exact ground state for comparison
eigenvalues = np.linalg.eigvalsh(H.simplify().to_matrix())
exact_energy = eigenvalues[0]

print(f"\nExact ground energy  : {exact_energy:.6f}")
print("Using COBYLA :")
print(f"Noisy VQE energy     : {best_noisy_energyC:.6f}")
print(f"Error due to noise   : {abs(best_noisy_energyC - exact_energy):.6f}")
print(f"Error %              : {abs(best_noisy_energyC - exact_energy)/abs(exact_energy)*100:.2f}%")


Exact ground energy  : -1.732051
Using COBYLA :
Noisy VQE energy     : -1.381836
Error due to noise   : 0.350215
Error %              : 20.22%


In [6]:
#Running on Noisy VQE using SPSA
optimizerS = SPSA(maxiter=300)
rngS = np.random.default_rng(42)

best_noisy_energyS = np.inf;
best_noisy_paramsS = None;

for i in range(5): #using 5 random restarts
    x0 = rngS.uniform(-np.pi, np.pi, ansatz.num_parameters)
    result = optimizerS.minimize(fun=expectation_value, x0=x0)
    if result.fun < best_noisy_energyS:
        best_noisy_energyS = result.fun
        best_noisy_parameterS = result.x

print("Using SPSA :")
print(f"Noisy VQE energy     : {best_noisy_energyS:.6f}")
print(f"Error due to noise   : {abs(best_noisy_energyS - exact_energy):.6f}")
print(f"Error %              : {abs(best_noisy_energyS - exact_energy)/abs(exact_energy)*100:.2f}%")

Using SPSA :
Noisy VQE energy     : -1.388672
Error due to noise   : 0.343379
Error %              : 19.82%
